# TinyChirp LEAF-Time TensorFlow

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import TYPE_CHECKING
from utils import (
    TARGET_AUDIO_LEN_TIME,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
    make_time_datasets,
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "leaf_tf"
paths = get_paths(MODEL_STEM)
BATCH_SIZE = 32
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs


In [ ]:
train_ds, val_ds, test_ds, label_names = make_time_datasets(batch_size = BATCH_SIZE)
num_labels = len(label_names)
print("Classes:", label_names)


In [ ]:
import numpy as np
import tensorflow as tf
import keras
from model_parts import GaborConv1D, GaussianPool1D, LogCompression

In [ ]:
NUM_FILTERS = 48
KERNEL_SIZE = 64
STRIDE = 16

def build_training_model(num_labels: int) -> keras.Model:
    inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))

    x = GaborConv1D(num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE, stride=STRIDE, name="gabor_conv")(inputs)
    x = GaussianPool1D(num_filters=NUM_FILTERS, pool_size=KERNEL_SIZE, stride=1, name="gauss_pool")(x)
    x = LogCompression(name="log_compress")(x)
    
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    outputs = keras.layers.Dense(num_labels, activation=None, name="dense_logits")(x)
    
    return keras.Model(inputs, outputs, name="leaf_fast_bird_model")


num_labels = 2 
training_model = build_training_model(num_labels)

# Use XLA compilation to speed up training
training_model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
    jit_compile=True
)

In [ ]:
training_model.summary()

In [ ]:
from utils import init_wandb, get_callbacks, finish_wandb

init_wandb(MODEL_STEM, config={
    "num_filters": NUM_FILTERS,
    "kernel_size": KERNEL_SIZE,
    "stride": STRIDE,
})

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    validation_steps=50,
    callbacks=get_callbacks(10,5,BATCH_SIZE),
)
finish_wandb()

In [ ]:
from model_parts import SquaredModulus

gabor_layer = training_model.get_layer("gabor_conv")
pool_layer = training_model.get_layer("gauss_pool")

baked_gabor = gabor_layer.get_filters().numpy()
baked_gauss = pool_layer.get_filters().numpy()

infer_inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1))

x = keras.layers.Conv1D(
    filters=NUM_FILTERS * 2,
    kernel_size=KERNEL_SIZE,
    strides=STRIDE,
    padding="same",
    use_bias=False,
    name="baked_gabor_conv",
)(infer_inputs)
x = SquaredModulus(name="squared_modulus")(x)
x = keras.layers.DepthwiseConv1D(
    kernel_size=KERNEL_SIZE,
    strides=1,
    padding="same",
    use_bias=False,
    name="baked_gauss_dw",
)(x)
x = LogCompression(name="log_compress")(x)

for layer in training_model.layers[4:]:
    x = layer(x)

inference_model = keras.Model(infer_inputs, x, name="leaf_fast_bird_inference")

inference_model.get_layer("baked_gabor_conv").set_weights([baked_gabor])
inference_model.get_layer("baked_gauss_dw").set_weights([baked_gauss])

for batch_audio, _ in test_ds.take(1):
    batch_audio_np = batch_audio.numpy()
    logits_train = training_model.predict(batch_audio_np, verbose=0)
    logits_infer = inference_model.predict(batch_audio_np, verbose=0)
    print(f"Max abs diff: {np.max(np.abs(logits_train - logits_infer)):.8f}")

In [ ]:
from utils import get_flops_native
flops = get_flops_native(inference_model, batch_size=1)
print(f"Total FLOPs: {flops}")

In [ ]:
rep_batches = build_representative_batches(test_ds, take=100)
export_keras_model_to_int8_tflite(inference_model, rep_batches, OUT_TFLITE)
print(f"Success! Wrote {OUT_TFLITE}")

In [ ]:
from utils import evaluate_tflite_model, SAMPLE_RATE

hyperparams = {
    "num_filters": NUM_FILTERS,
    "kernel_size": KERNEL_SIZE,
    "stride": STRIDE,
    "dense_hidden": 64,
    "target_audio_len": TARGET_AUDIO_LEN_TIME,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")